# [Semantic Kernel 04 - Single Agent with native plugin](https://github.com/microsoft/semantic-kernel/blob/main/python/samples/getting_started_with_agents/step7_assistant.py)
The following sample demonstrates how to create an OpenAI assistant using either Azure OpenAI or OpenAI.<br/>
OpenAI Assistants allow for function calling, the use of file search and a code interpreter.<br/>
Assistant Threads are used to manage the conversation state, similar to a Semantic Kernel Chat History. 

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior # Auto(), Required() or NoneInvoke()
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.agents.open_ai import AzureAssistantAgent
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents.chat_message_content import ChatMessageContent

load_dotenv("./../config/credentials_my.env")

agent_name                  = "agent_name"
chatcompletion_service_id   = "chatcompletion_service_id"
instructions                = "you are a clever agent"
content                     = "Toggle the status of my second light."

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Define the Kernel

In [2]:
kernel = Kernel()
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001F56D6BDD30>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [3]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)

# Create an AzureChatCompletion AI Service and add it to the Kernel

In [4]:
kernel.add_service(AzureChatCompletion(service_id=chatcompletion_service_id))
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='chatcompletion_service_id', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001F5470E7E00>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001F56D6BDD30>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Define a native plugin, then add it to the kernel

In [5]:
# First, we define the plugin through its class...

class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
    
    lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": True},
    ]

    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights

    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

In [6]:
# ...then, we add the plugin to the kernel, using a new plugin name

kernel.add_plugin(
    plugin=LightsPlugin(),
    plugin_name="Lights",
)

KernelPlugin(name='Lights', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='Lights', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, schema_data={'type': 'integer'}, include_in_function_choices=True), KernelParameterMetadata(name='is_on', description=None, default_value=None, type_='bool', is_required=True, type_object=<class 'bool'>, schema_data={'type': 'boolean'}, include_in_function_choices=True)], is_prompt=False, is_asynchronous=False, return_parameter=KernelParameterMetadata(name='return', description='the output is a string', default_value=None, type_='str', is_required=True, type_object=<class 'str'>, schema_data={'type': 'string', 'description': 'the output is a string'}, include_in_function_choices=True), additional_properties={}), invocation_duratio

# Create the Azure Assistant Agent

In [7]:
agent = await AzureAssistantAgent.create(
    kernel=kernel, service_id=chatcompletion_service_id, name=agent_name, instructions=instructions
)

agent

AzureAssistantAgent(id='274399bd-6b6a-4aef-bc52-9c47b871b9d7', description=None, name='agent_name', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='chatcompletion_service_id', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001F5470E7E00>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001F56D6BDD30>, plugins={'Lights': KernelPlugin(name='Lights', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='Lights', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, sc

# Create Thread and add a message to it

In [8]:
thread_id = await agent.create_thread()
thread_id

'thread_kyMNwxtdT9etcPjixg3RN3xm'

In [9]:
message = ChatMessageContent(role=AuthorRole.USER, content=content)

await agent.add_chat_message(thread_id=thread_id, message=message)

i =0
async for message in agent.get_thread_messages(thread_id):
    i += 1
    print(f">>> Message {i} - {message}\n")

>>> Message 1 - Toggle the status of my second light.



# Run the agent

In [10]:
async for message in agent.invoke(thread_id=thread_id):
    print(message)

The status of your second light (Porch light) has been toggled. It is now ON.


# Check the message history through the thread id
**IMPORTANT**
- The Assistant Agent automatically manages the history through the thread.
- The list of message history in the thread starts with the most recent one to the oldest one.

In [11]:
i =0
async for message in agent.get_thread_messages(thread_id):
    i += 1
    print(f">>> Message {i} - {message}\n")

>>> Message 1 - The status of your second light (Porch light) has been toggled. It is now ON.

>>> Message 2 - Toggle the status of my second light.



# Additional tests. Run multiple times to toggle the first light.

In [12]:
message = ChatMessageContent(role=AuthorRole.USER, content="Toggle the first light and give me the status of all my lights.")
await agent.add_chat_message(thread_id=thread_id, message=message)
i =0
async for message in agent.get_thread_messages(thread_id):
    i += 1
    print(f">>> Message {i} - {message}\n")

>>> Message 1 - Toggle the first light and give me the status of all my lights.

>>> Message 2 - The status of your second light (Porch light) has been toggled. It is now ON.

>>> Message 3 - Toggle the status of my second light.



In [13]:
async for message in agent.invoke(thread_id=thread_id):
    print(message)

The first light (Table Lamp) has been toggled. Here is the status of all your lights:

1. Table Lamp: ON
2. Porch light: ON
3. Chandelier: ON


In [14]:
i =0
async for message in agent.get_thread_messages(thread_id):
    i += 1
    print(f">>> Message {i} - {message}\n")

>>> Message 1 - The first light (Table Lamp) has been toggled. Here is the status of all your lights:

1. Table Lamp: ON
2. Porch light: ON
3. Chandelier: ON

>>> Message 2 - Toggle the first light and give me the status of all my lights.

>>> Message 3 - The status of your second light (Porch light) has been toggled. It is now ON.

>>> Message 4 - Toggle the status of my second light.

